# Clue Utilization & Early Termination Rate Analysis

## Objective
Compute and analyze two key collaboration metrics:
1. **Clue Utilization**: ratio of correct guesses to the specified clue number `n`
2. **Early Termination Rate**: proportion of turns where Guessers yield before exhausting allowed guesses

## Data Source
- `game_logs/`: Raw game results from persona pairs

## Configuration
- **MAX_PERSONA_ID**: Configurable limit to match experiment settings
- **LOGS_DIR**: Directory containing game log files

## 1. Setup and Imports

In [ ]:
import json
import os
import glob
import re
from collections import defaultdict
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, ttest_ind, mannwhitneyu

# Import unified loading functions
from analysis_utils import load_game_logs_with_turns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ===== LOGGING SETUP =====
LOG_FILE = 'metrics.txt'

def clear_log():
    """Clear the log file at start of notebook run."""
    with open(LOG_FILE, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("CLUE UTILIZATION & EARLY TERMINATION RATE ANALYSIS\n")
        f.write("=" * 80 + "\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("This file contains all numerical results and statistics from metrics.ipynb.\n")
        f.write("For plots and visualizations, see the figures/ directory.\n")
        f.write("=" * 80 + "\n\n")

def log_print(*args, sep=' ', end='\n'):
    """Print to console and append to log file."""
    text = sep.join(str(arg) for arg in args)
    print(text, end=end)
    with open(LOG_FILE, 'a') as f:
        f.write(text + end)

# Clear log file at notebook start
clear_log()

# ===== CONFIGURATION =====
MAX_PERSONA_ID = 10  # Set to match your experiment configuration
LOGS_DIR = 'game_logs/tailored_words_to_persona1'  # Directory containing game logs
log_print(f"Analysis configured for personas 1-{MAX_PERSONA_ID}")
log_print(f"Loading logs from: {LOGS_DIR}")

## 2. Load and Parse Game Logs

In [ ]:
# parse_game_filename is now provided by analysis_utils module
# This cell is kept for backward compatibility but the function is imported

In [ ]:
# Load data using unified loading function from analysis_utils
# This function auto-detects the log format (old or new) and handles both

games_df, turns_df = load_game_logs_with_turns(LOGS_DIR, MAX_PERSONA_ID)
log_print(f"\nGames shape: {games_df.shape}")
log_print(f"Turns shape: {turns_df.shape}")
turns_df.head(10)

## 3. Clue Utilization Analysis

**Definition**: Clue utilization = correct guesses / clue number (n)

This metric measures how effectively the Guesser converts the Codemaster's clues into correct guesses.

In [4]:
# Overall clue utilization statistics
log_print("\n" + "=" * 80)
log_print("SECTION 3: CLUE UTILIZATION ANALYSIS")
log_print("=" * 80)
log_print("\nDefinition: Clue utilization = correct guesses / clue number (n)")
log_print("This metric measures how effectively the Guesser converts the Codemaster's")
log_print("clues into correct guesses. A value of 1.0 means the Guesser correctly")
log_print("identified exactly as many words as the Codemaster indicated.\n")

log_print("=== Clue Utilization Statistics (Per Turn) ===")
log_print(f"Sample size (n): {len(turns_df)} turns")
log_print(f"Mean: {turns_df['clue_utilization'].mean():.4f}")
log_print(f"Median: {turns_df['clue_utilization'].median():.4f}")
log_print(f"Std: {turns_df['clue_utilization'].std():.4f}")
log_print(f"Min: {turns_df['clue_utilization'].min():.4f}")
log_print(f"Max: {turns_df['clue_utilization'].max():.4f}")

log_print(f"\nInterpretation: On average, Guessers correctly identified {turns_df['clue_utilization'].mean()*100:.1f}%")
log_print(f"of the words indicated by the Codemaster's clue number.")

# Distribution of clue numbers
log_print(f"\n=== Clue Number Distribution ===")
log_print("(How many words Codemasters typically indicate per clue)")
clue_dist = turns_df['clue_number'].value_counts().sort_index()
for clue_num, count in clue_dist.items():
    pct = count / len(turns_df) * 100
    log_print(f"  n={clue_num}: {count} turns ({pct:.1f}%)")


SECTION 3: CLUE UTILIZATION ANALYSIS

Definition: Clue utilization = correct guesses / clue number (n)
This metric measures how effectively the Guesser converts the Codemaster's
clues into correct guesses. A value of 1.0 means the Guesser correctly
identified exactly as many words as the Codemaster indicated.

=== Clue Utilization Statistics (Per Turn) ===
Sample size (n): 0 turns


KeyError: 'clue_utilization'

In [ ]:
# Histogram of clue utilization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of clue utilization
ax1 = axes[0]
ax1.hist(turns_df['clue_utilization'], bins=20, edgecolor='black', alpha=0.7)
ax1.axvline(turns_df['clue_utilization'].mean(), color='red', linestyle='--', 
            label=f"Mean: {turns_df['clue_utilization'].mean():.3f}")
ax1.axvline(1.0, color='green', linestyle=':', label='Perfect utilization (1.0)')
ax1.set_xlabel('Clue Utilization', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Clue Utilization (Per Turn)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Clue utilization by clue number
ax2 = axes[1]
clue_util_by_n = turns_df.groupby('clue_number')['clue_utilization'].agg(['mean', 'std', 'count'])
clue_util_by_n = clue_util_by_n[clue_util_by_n['count'] >= 5]  # Filter low counts
ax2.bar(clue_util_by_n.index, clue_util_by_n['mean'], yerr=clue_util_by_n['std'], 
        capsize=5, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Clue Number (n)', fontsize=12)
ax2.set_ylabel('Mean Clue Utilization', fontsize=12)
ax2.set_title('Clue Utilization by Clue Number', fontsize=14)
ax2.set_xticks(clue_util_by_n.index)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/clue_utilization_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Aggregate clue utilization by persona pair
# Filter to only include pairs where both have personas
turns_with_personas = turns_df[
    (turns_df['cm_id'].notna()) & (turns_df['guesser_id'].notna())
].copy()

clue_util_by_pair = turns_with_personas.groupby(['cm_id', 'guesser_id']).agg({
    'clue_utilization': ['mean', 'std', 'count'],
    'correct_guesses': 'sum',
    'clue_number': 'sum'
}).reset_index()

clue_util_by_pair.columns = ['cm_id', 'guesser_id', 'mean_clue_util', 'std_clue_util', 
                             'num_turns', 'total_correct', 'total_clue_numbers']

# Add aggregate efficiency (total correct / total clue numbers)
clue_util_by_pair['aggregate_efficiency'] = (
    clue_util_by_pair['total_correct'] / clue_util_by_pair['total_clue_numbers']
)

# Add same persona flag
clue_util_by_pair['is_same_persona'] = clue_util_by_pair['cm_id'] == clue_util_by_pair['guesser_id']

log_print(f"\n=== Clue Utilization by Persona Pair ===")
log_print(f"Aggregated clue utilization for {len(clue_util_by_pair)} persona pairs")

log_print(f"\nTop 5 pairs by mean clue utilization:")
top5 = clue_util_by_pair.sort_values('mean_clue_util', ascending=False).head(5)
for _, row in top5.iterrows():
    log_print(f"  CM {int(row['cm_id'])} -> Guesser {int(row['guesser_id'])}: {row['mean_clue_util']:.3f} (n={int(row['num_turns'])} turns)")

clue_util_by_pair.sort_values('mean_clue_util', ascending=False).head(10)

In [ ]:
# Create heatmap of clue utilization by persona pair
persona_ids = sorted(turns_with_personas['cm_id'].dropna().unique())
n_personas = len(persona_ids)

clue_util_matrix = np.full((n_personas, n_personas), np.nan)
for _, row in clue_util_by_pair.iterrows():
    i = int(row['cm_id']) - 1
    j = int(row['guesser_id']) - 1
    if 0 <= i < n_personas and 0 <= j < n_personas:
        clue_util_matrix[i, j] = row['mean_clue_util']

clue_util_heatmap_df = pd.DataFrame(
    clue_util_matrix,
    index=[f'cm{i}' for i in range(1, n_personas + 1)],
    columns=[f'g{i}' for i in range(1, n_personas + 1)]
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    clue_util_heatmap_df,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1.5,
    cbar_kws={'label': 'Mean Clue Utilization'},
    square=True,
    linewidths=0.5,
    linecolor='white'
)
plt.title('Clue Utilization by Persona Pair', fontsize=14, pad=20)
plt.xlabel('Guesser ID', fontsize=12)
plt.ylabel('Codemaster ID', fontsize=12)
plt.tight_layout()
plt.savefig('figures/clue_utilization_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 4. Early Termination Rate Analysis

**Definition**: Proportion of turns where the Guesser yields before exhausting allowed guesses (n+1).

This metric serves as a proxy for calibrated confidence - Guessers who stop early may have better uncertainty estimation.

In [ ]:
# Overall early termination statistics
log_print("\n" + "=" * 80)
log_print("SECTION 4: EARLY TERMINATION RATE ANALYSIS")
log_print("=" * 80)
log_print("\nDefinition: Proportion of turns where the Guesser yields before exhausting")
log_print("allowed guesses (n+1). This metric serves as a proxy for calibrated confidence -")
log_print("Guessers who stop early may have better uncertainty estimation.\n")

log_print("=== Early Termination Statistics ===")
total_turns = len(turns_df)
early_term_turns = turns_df['turn_ended_early'].sum()
early_term_rate = early_term_turns / total_turns

log_print(f"Total turns: {total_turns}")
log_print(f"Turns ended early: {early_term_turns}")
log_print(f"Early termination rate: {early_term_rate:.4f} ({early_term_rate*100:.1f}%)")

log_print(f"\nInterpretation: {early_term_rate*100:.1f}% of turns ended with the Guesser")
log_print("voluntarily stopping before using all allowed guesses.")

# Breakdown by turn outcome
log_print(f"\n=== Early Termination by Turn Outcome ===")
log_print("(How often Guessers ended early based on what happened)")
early_term_by_outcome = turns_df.groupby('turn_outcome')['turn_ended_early'].agg(['sum', 'count', 'mean'])
early_term_by_outcome.columns = ['early_term_count', 'total_count', 'early_term_rate']
for outcome, row in early_term_by_outcome.iterrows():
    log_print(f"  {outcome}: {row['early_term_rate']*100:.1f}% early termination ({int(row['early_term_count'])}/{int(row['total_count'])} turns)")

print(early_term_by_outcome)

In [ ]:
# Aggregate early termination rate by persona pair
early_term_by_pair = turns_with_personas.groupby(['cm_id', 'guesser_id']).agg({
    'turn_ended_early': ['sum', 'count', 'mean']
}).reset_index()

early_term_by_pair.columns = ['cm_id', 'guesser_id', 'early_term_count', 
                               'total_turns', 'early_term_rate']

# Add same persona flag
early_term_by_pair['is_same_persona'] = early_term_by_pair['cm_id'] == early_term_by_pair['guesser_id']

log_print(f"\n=== Early Termination by Persona Pair ===")
log_print(f"Aggregated early termination for {len(early_term_by_pair)} persona pairs")

log_print(f"\nTop 5 pairs by early termination rate:")
top5_et = early_term_by_pair.sort_values('early_term_rate', ascending=False).head(5)
for _, row in top5_et.iterrows():
    log_print(f"  CM {int(row['cm_id'])} -> Guesser {int(row['guesser_id'])}: {row['early_term_rate']*100:.1f}% ({int(row['early_term_count'])}/{int(row['total_turns'])} turns)")

early_term_by_pair.sort_values('early_term_rate', ascending=False).head(10)

In [ ]:
# Create heatmap of early termination rate by persona pair
early_term_matrix = np.full((n_personas, n_personas), np.nan)
for _, row in early_term_by_pair.iterrows():
    i = int(row['cm_id']) - 1
    j = int(row['guesser_id']) - 1
    if 0 <= i < n_personas and 0 <= j < n_personas:
        early_term_matrix[i, j] = row['early_term_rate']

early_term_heatmap_df = pd.DataFrame(
    early_term_matrix,
    index=[f'cm{i}' for i in range(1, n_personas + 1)],
    columns=[f'g{i}' for i in range(1, n_personas + 1)]
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    early_term_heatmap_df,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Early Termination Rate'},
    square=True,
    linewidths=0.5,
    linecolor='white'
)
plt.title('Early Termination Rate by Persona Pair', fontsize=14, pad=20)
plt.xlabel('Guesser ID', fontsize=12)
plt.ylabel('Codemaster ID', fontsize=12)
plt.tight_layout()
plt.savefig('figures/early_termination_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization: Early termination vs clue utilization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of early termination by turn outcome
ax1 = axes[0]
outcome_counts = turns_df.groupby(['turn_outcome', 'turn_ended_early']).size().unstack(fill_value=0)
outcome_counts.plot(kind='bar', ax=ax1, color=['steelblue', 'coral'])
ax1.set_xlabel('Turn Outcome', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Early Termination by Turn Outcome', fontsize=14)
ax1.legend(['Continued', 'Ended Early'], title='Turn Ended Early')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3, axis='y')

# Scatter: Early termination rate vs clue utilization (per persona pair)
ax2 = axes[1]
merged_pair_data = pd.merge(
    clue_util_by_pair[['cm_id', 'guesser_id', 'mean_clue_util', 'is_same_persona']],
    early_term_by_pair[['cm_id', 'guesser_id', 'early_term_rate']],
    on=['cm_id', 'guesser_id']
)

colors = merged_pair_data['is_same_persona'].map({True: 'red', False: 'blue'})
ax2.scatter(merged_pair_data['early_term_rate'], merged_pair_data['mean_clue_util'], 
            c=colors, alpha=0.6, s=80)
ax2.set_xlabel('Early Termination Rate', fontsize=12)
ax2.set_ylabel('Mean Clue Utilization', fontsize=12)
ax2.set_title('Early Termination vs Clue Utilization', fontsize=14)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', label='Same Persona'),
                   Patch(facecolor='blue', label='Different Persona')]
ax2.legend(handles=legend_elements)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/early_termination_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

Test relationships between metrics and game outcomes.

In [ ]:
# Aggregate game-level metrics
log_print("\n" + "=" * 80)
log_print("SECTION 5: CORRELATION ANALYSIS")
log_print("=" * 80)
log_print("\nTesting relationships between collaboration metrics and game outcomes.\n")

game_metrics = games_df[
    (games_df['cm_id'].notna()) & (games_df['guesser_id'].notna())
].copy()

# Aggregate turn metrics per game
turn_agg = turns_with_personas.groupby('game_id').agg({
    'clue_utilization': 'mean',
    'turn_ended_early': 'mean',  # This gives the rate
    'correct_guesses': 'sum',
    'clue_number': 'sum'
}).reset_index()
turn_agg.columns = ['game_id', 'game_clue_util', 'game_early_term_rate', 
                    'total_correct', 'total_clue_numbers']

# Merge with game outcomes
game_analysis = pd.merge(game_metrics, turn_agg, on='game_id', how='left')
game_analysis['is_same_persona'] = game_analysis['cm_id'] == game_analysis['guesser_id']

log_print(f"Games with metrics: {len(game_analysis)}")
log_print(f"Games with same persona: {game_analysis['is_same_persona'].sum()}")
log_print(f"Games with different persona: {(~game_analysis['is_same_persona']).sum()}")
game_analysis.head()

In [ ]:
def compute_correlation(df, x_col, y_col, label=""):
    """Compute Pearson and Spearman correlations with p-values."""
    valid_data = df[[x_col, y_col]].dropna()
    
    if len(valid_data) < 3:
        log_print(f"Insufficient data for correlation: {label}")
        return None
    
    x = valid_data[x_col]
    y = valid_data[y_col]
    
    pearson_r, pearson_p = pearsonr(x, y)
    spearman_r, spearman_p = spearmanr(x, y)
    
    sig_pearson = '***' if pearson_p < 0.001 else '**' if pearson_p < 0.01 else '*' if pearson_p < 0.05 else '(n.s.)'
    sig_spearman = '***' if spearman_p < 0.001 else '**' if spearman_p < 0.01 else '*' if spearman_p < 0.05 else '(n.s.)'
    
    log_print(f"\n{'='*60}")
    log_print(f"{label}")
    log_print(f"{'='*60}")
    log_print(f"Sample size: {len(valid_data)}")
    log_print(f"Pearson:  r = {pearson_r:+.4f}, p = {pearson_p:.4f} {sig_pearson}")
    log_print(f"Spearman: ρ = {spearman_r:+.4f}, p = {spearman_p:.4f} {sig_spearman}")
    
    return {'x': x_col, 'y': y_col, 'n': len(valid_data), 
            'pearson_r': pearson_r, 'pearson_p': pearson_p,
            'spearman_r': spearman_r, 'spearman_p': spearman_p}

# Convert 'won' to numeric
game_analysis['won_numeric'] = game_analysis['won'].astype(int)

log_print("\n=== Correlation Tests ===")
log_print("Significance levels: *** p<0.001, ** p<0.01, * p<0.05, (n.s.) not significant")

# Correlation: Clue utilization vs Win
corr1 = compute_correlation(game_analysis, 'game_clue_util', 'won_numeric', 
                            "Clue Utilization → Win")
log_print("\nInterpretation: Tests whether higher clue utilization predicts game wins.")

# Correlation: Early termination rate vs Win
corr2 = compute_correlation(game_analysis, 'game_early_term_rate', 'won_numeric',
                            "Early Termination Rate → Win")
log_print("\nInterpretation: Tests whether stopping early (calibrated confidence) predicts wins.")

# Correlation: Clue utilization vs Total turns (efficiency)
corr3 = compute_correlation(game_analysis, 'game_clue_util', 'total_turns',
                            "Clue Utilization → Total Turns")
log_print("\nInterpretation: Tests whether better clue utilization leads to shorter games.")

# Correlation: Early termination vs Clue utilization
corr4 = compute_correlation(game_analysis, 'game_early_term_rate', 'game_clue_util',
                            "Early Termination Rate ↔ Clue Utilization")
log_print("\nInterpretation: Tests the relationship between these two collaboration metrics.")

In [ ]:
# Compare same persona vs different persona pairs
log_print("\n" + "=" * 80)
log_print("SAME vs DIFFERENT PERSONA COMPARISON")
log_print("=" * 80)
log_print("\nComparing game performance when Codemaster and Guesser share the same")
log_print("persona vs. when they have different personas.\n")

same_persona = game_analysis[game_analysis['is_same_persona'] == True]
diff_persona = game_analysis[game_analysis['is_same_persona'] == False]

log_print(f"Same Persona Games (n={len(same_persona)}):")
log_print(f"  Win Rate: {same_persona['won'].mean():.3f} ({same_persona['won'].mean()*100:.1f}%)")
log_print(f"  Clue Utilization: {same_persona['game_clue_util'].mean():.3f} ± {same_persona['game_clue_util'].std():.3f}")
log_print(f"  Early Term Rate: {same_persona['game_early_term_rate'].mean():.3f} ± {same_persona['game_early_term_rate'].std():.3f}")

log_print(f"\nDifferent Persona Games (n={len(diff_persona)}):")
log_print(f"  Win Rate: {diff_persona['won'].mean():.3f} ({diff_persona['won'].mean()*100:.1f}%)")
log_print(f"  Clue Utilization: {diff_persona['game_clue_util'].mean():.3f} ± {diff_persona['game_clue_util'].std():.3f}")
log_print(f"  Early Term Rate: {diff_persona['game_early_term_rate'].mean():.3f} ± {diff_persona['game_early_term_rate'].std():.3f}")

# Statistical tests
if len(same_persona) > 0 and len(diff_persona) > 0:
    t_clue, p_clue = ttest_ind(same_persona['game_clue_util'].dropna(), 
                                diff_persona['game_clue_util'].dropna())
    t_early, p_early = ttest_ind(same_persona['game_early_term_rate'].dropna(),
                                  diff_persona['game_early_term_rate'].dropna())
    
    sig_clue = '***' if p_clue < 0.001 else '**' if p_clue < 0.01 else '*' if p_clue < 0.05 else '(n.s.)'
    sig_early = '***' if p_early < 0.001 else '**' if p_early < 0.01 else '*' if p_early < 0.05 else '(n.s.)'
    
    log_print(f"\n=== Statistical Tests (Independent t-test) ===")
    log_print(f"Clue Utilization: t={t_clue:.3f}, p={p_clue:.4f} {sig_clue}")
    log_print(f"Early Term Rate: t={t_early:.3f}, p={p_early:.4f} {sig_early}")
    
    log_print(f"\nInterpretation:")
    if p_clue < 0.05:
        direction = "higher" if same_persona['game_clue_util'].mean() > diff_persona['game_clue_util'].mean() else "lower"
        log_print(f"  - Same-persona pairs have significantly {direction} clue utilization.")
    else:
        log_print(f"  - No significant difference in clue utilization between groups.")
    
    if p_early < 0.05:
        direction = "higher" if same_persona['game_early_term_rate'].mean() > diff_persona['game_early_term_rate'].mean() else "lower"
        log_print(f"  - Same-persona pairs have significantly {direction} early termination rate.")

In [ ]:
# Visualization: Scatter plots with regression lines
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Clue utilization by win/loss
ax1 = axes[0]
game_analysis.boxplot(column='game_clue_util', by='won', ax=ax1)
ax1.set_xlabel('Game Won', fontsize=12)
ax1.set_ylabel('Clue Utilization', fontsize=12)
ax1.set_title('Clue Utilization by Game Outcome', fontsize=14)
plt.suptitle('')  # Remove automatic title

# Plot 2: Early termination rate by win/loss
ax2 = axes[1]
game_analysis.boxplot(column='game_early_term_rate', by='won', ax=ax2)
ax2.set_xlabel('Game Won', fontsize=12)
ax2.set_ylabel('Early Termination Rate', fontsize=12)
ax2.set_title('Early Termination Rate by Game Outcome', fontsize=14)
plt.suptitle('')

# Plot 3: Comparison by persona type
ax3 = axes[2]
metrics_comparison = pd.DataFrame({
    'Clue Util (Same)': [same_persona['game_clue_util'].mean()],
    'Clue Util (Diff)': [diff_persona['game_clue_util'].mean()],
    'Early Term (Same)': [same_persona['game_early_term_rate'].mean()],
    'Early Term (Diff)': [diff_persona['game_early_term_rate'].mean()]
})

x_pos = [0, 1, 3, 4]
colors = ['lightblue', 'lightcoral', 'lightblue', 'lightcoral']
values = [same_persona['game_clue_util'].mean(), diff_persona['game_clue_util'].mean(),
          same_persona['game_early_term_rate'].mean(), diff_persona['game_early_term_rate'].mean()]
bars = ax3.bar(x_pos, values, color=colors, edgecolor='black')
ax3.set_xticks([0.5, 3.5])
ax3.set_xticklabels(['Clue Utilization', 'Early Term Rate'])
ax3.set_ylabel('Mean Value', fontsize=12)
ax3.set_title('Metrics: Same vs Different Persona', fontsize=14)
ax3.legend([bars[0], bars[1]], ['Same Persona', 'Different Persona'])
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/metrics_correlation_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
# Create comprehensive summary table by persona pair
log_print("\n" + "=" * 80)
log_print("SECTION 6: SUMMARY STATISTICS")
log_print("=" * 80)

# Aggregate game outcomes
game_agg = game_analysis.groupby(['cm_id', 'guesser_id']).agg({
    'won': ['sum', 'count', 'mean'],
    'total_turns': 'mean',
    'game_clue_util': 'mean',
    'game_early_term_rate': 'mean'
}).reset_index()

game_agg.columns = ['cm_id', 'guesser_id', 'wins', 'total_games', 'win_rate',
                    'avg_turns', 'avg_clue_util', 'avg_early_term_rate']

game_agg['is_same_persona'] = game_agg['cm_id'] == game_agg['guesser_id']

log_print("\n=== Summary Statistics by Persona Pair ===")
log_print(f"Total persona pairs: {len(game_agg)}")
log_print(f"Same persona pairs: {game_agg['is_same_persona'].sum()}")
log_print(f"Different persona pairs: {(~game_agg['is_same_persona']).sum()}")

log_print(f"\n=== Overall Metrics (across all pairs) ===")
log_print(f"Mean Win Rate: {game_agg['win_rate'].mean():.3f} ± {game_agg['win_rate'].std():.3f}")
log_print(f"Mean Clue Utilization: {game_agg['avg_clue_util'].mean():.3f} ± {game_agg['avg_clue_util'].std():.3f}")
log_print(f"Mean Early Term Rate: {game_agg['avg_early_term_rate'].mean():.3f} ± {game_agg['avg_early_term_rate'].std():.3f}")

# Display top performing pairs
log_print(f"\n=== Top 10 Pairs by Win Rate ===")
top10_win = game_agg.sort_values('win_rate', ascending=False).head(10)
for _, row in top10_win.iterrows():
    same_flag = " [SAME]" if row['is_same_persona'] else ""
    log_print(f"  CM {int(row['cm_id'])} -> Guesser {int(row['guesser_id'])}: {row['win_rate']*100:.0f}% win rate ({int(row['wins'])}/{int(row['total_games'])} games){same_flag}")

log_print("\n" + "=" * 80)
log_print("END OF METRICS ANALYSIS")
log_print("=" * 80)

game_agg.sort_values('win_rate', ascending=False).head(10)